# Diabetes Prediction – Model Comparison with GridSearchCV
Converted from the training script.

In [ ]:
import os
import pickle
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
# ==========================================
# 1. LOAD DATASET
# ==========================================

df = pd.read_csv("dataset/kaggle_diabetes.csv")

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

In [ ]:
# ==========================================
# 2. SPLIT DATASET
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# ==========================================
# 3. DEFINE MODELS AND PARAMETERS
# ==========================================

models = {

    "Logistic Regression": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=1000))
        ]),
        "params": {
            "classifier__C": [0.1, 1, 10],
            "classifier__solver": ["liblinear", "lbfgs"]
        }
    },

    "K-Nearest Neighbors": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", KNeighborsClassifier())
        ]),
        "params": {
            "classifier__n_neighbors": [3, 5, 7, 9, 11],
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"]
        }
    },

    "Support Vector Machine": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", SVC())
        ]),
        "params": {
            "classifier__C": [0.1, 1, 10],
            "classifier__kernel": ["linear", "rbf"],
            "classifier__gamma": ["scale", "auto"]
        }
    },

    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "criterion": ["gini", "entropy"],
            "max_depth": [None, 5, 10, 20],
            "min_samples_split": [2, 5, 10]
        }
    },

    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
            "max_depth": [None, 5, 10, 20],
            "criterion": ["gini", "entropy"]
        }
    },

    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.01, 0.1, 0.2],
            "max_depth": [2, 3, 5]
        }
    }
}

In [ ]:
# ==========================================
# 4. GRID SEARCH CV
# ==========================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

best_model = None
best_model_name = None
best_score = -1

for name, config in models.items():

    print("\n" + "=" * 50)
    print("Training:", name)
    print("=" * 50)

    grid = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    model = grid.best_estimator_

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print("Best Parameters:", grid.best_params_)
    print("CV Accuracy:", grid.best_score_)
    print("Test Accuracy:", accuracy)

    results.append({
        "Model": name,
        "CV Accuracy": grid.best_score_,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    if grid.best_score_ > best_score:

        best_score = grid.best_score_
        best_model = model
        best_model_name = name

In [ ]:
# ==========================================
# 5. COMPARE ALL MODELS
# ==========================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="CV Accuracy",
    ascending=False
)

print("\n\nFINAL MODEL COMPARISON")
print(results_df.to_string(index=False))

In [ ]:
# ==========================================
# 6. SAVE RESULTS
# ==========================================

os.makedirs("experiments", exist_ok=True)

results_df.to_csv(
    "experiments/model_comparison.csv",
    index=False
)

In [ ]:
# ==========================================
# 7. BEST MODEL EVALUATION
# ==========================================

print("\nBEST MODEL:", best_model_name)
print("\nClassification Report:")
print(classification_report(y_test, best_model.predict(X_test)))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, best_model.predict(X_test)))

In [ ]:
# ==========================================
# 8. SAVE BEST MODEL
# ==========================================

model_path = "experiments/best_diabetes_model.pkl"

with open(model_path, "wb") as file:
    pickle.dump(best_model, file)

print("\nBest model saved successfully!")
print("Model:", best_model_name)
print("Location:", model_path)

## Model Comparison – Visualizations
Run these cells after the grid-search cells above (they use `results_df`, `best_model`, `best_model_name`, `X_test`, `y_test`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

plot_df = results_df.reset_index(drop=True)
model_names = plot_df["Model"].tolist()
metrics = ["Accuracy", "Precision", "Recall", "F1 Score"]

In [ ]:
# 1. Grouped bar chart: test-set metrics for every model
x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))
for i, metric in enumerate(metrics):
    bars = ax.bar(x + (i - 1.5) * width, plot_df[metric], width, label=metric)
    ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2)

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=20, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.1)
ax.set_title("Model Comparison – Test Set Metrics")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.25))
plt.tight_layout()
plt.savefig("experiments/model_comparison_metrics.png", bbox_inches="tight")
plt.show()

In [ ]:
# 2. Cross-validation accuracy vs test accuracy
x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - width / 2, plot_df["CV Accuracy"], width, label="CV Accuracy")
b2 = ax.bar(x + width / 2, plot_df["Accuracy"], width, label="Test Accuracy")
ax.bar_label(b1, fmt="%.3f", fontsize=8, padding=2)
ax.bar_label(b2, fmt="%.3f", fontsize=8, padding=2)

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=20, ha="right")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Cross-Validation vs Test Accuracy")
ax.legend()
plt.tight_layout()
plt.savefig("experiments/cv_vs_test_accuracy.png", bbox_inches="tight")
plt.show()

In [ ]:
# 3. Heatmap of all metrics for all models
heat_cols = ["CV Accuracy"] + metrics
heat_data = plot_df[heat_cols].to_numpy()

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(heat_data, cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(heat_cols)))
ax.set_xticklabels(heat_cols, rotation=20, ha="right")
ax.set_yticks(range(len(model_names)))
ax.set_yticklabels(model_names)

for i in range(heat_data.shape[0]):
    for j in range(heat_data.shape[1]):
        val = heat_data[i, j]
        ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                color="white" if val > 0.6 else "black", fontsize=9)

ax.set_title("Metric Heatmap (all models)")
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
plt.tight_layout()
plt.savefig("experiments/metrics_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# 4. Models ranked by CV accuracy (horizontal bars, best on top)
ranked = plot_df.sort_values("CV Accuracy", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(ranked["Model"], ranked["CV Accuracy"])
ax.bar_label(bars, fmt="%.4f", padding=3)
ax.set_xlim(0, 1.05)
ax.set_xlabel("CV Accuracy")
ax.set_title("Model Ranking by Cross-Validation Accuracy")
bars[-1].set_edgecolor("black")
bars[-1].set_linewidth(2)
plt.tight_layout()
plt.savefig("experiments/model_ranking.png", bbox_inches="tight")
plt.show()

In [ ]:
# 5. Best model: confusion matrix + ROC curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test,
    display_labels=["No Diabetes", "Diabetes"],
    cmap="Blues", ax=axes[0], colorbar=False
)
axes[0].set_title(f"Confusion Matrix – {best_model_name}")

if hasattr(best_model, "predict_proba"):
    scores = best_model.predict_proba(X_test)[:, 1]
else:
    scores = best_model.decision_function(X_test)

fpr, tpr, _ = roc_curve(y_test, scores)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title(f"ROC Curve – {best_model_name}")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.savefig("experiments/best_model_confusion_roc.png", bbox_inches="tight")
plt.show()